In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ChairNonchair.zip to ChairNonchair.zip


In [ ]:
import zipfile
import os
import cv2
import h5py
import numpy as np

In [ ]:
with zipfile.ZipFile("ChairNonchair.zip", 'r') as zip_ref:
    zip_ref.extractall()

print("Dataset extracted")

Dataset extracted


In [ ]:
print(os.listdir("ChairNonchair"))

['Nonchair', 'Chair']


In [ ]:
IMG_SIZE = 128

images = []
labels = []

CHAIR_FOLDER = "ChairNonchair/Chair"
NON_CHAIR_FOLDER = "ChairNonchair/Nonchair"

for file in os.listdir(CHAIR_FOLDER):

    path = os.path.join(CHAIR_FOLDER, file)

    img = cv2.imread(path)

    if img is None:
        continue

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    images.append(img)
    labels.append(1)


for file in os.listdir(NON_CHAIR_FOLDER):

    path = os.path.join(NON_CHAIR_FOLDER, file)

    img = cv2.imread(path)

    if img is None:
        continue

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    images.append(img)
    labels.append(0)

X = np.array(images, dtype=np.float32)
Y = np.array(labels)

X = X / 255.0

print("X shape:", X.shape)
print("Y shape:", Y.shape)

with h5py.File("data.h5", "w") as hf:

    hf.create_dataset("X", data=X)
    hf.create_dataset("Y", data=Y)

print("data.h5 created successfully")

X shape: (165, 128, 128, 3)
Y shape: (165,)
data.h5 created successfully


In [ ]:
with h5py.File("data.h5", "r") as hf:

    X = hf["X"][:]
    Y = hf["Y"][:]

print(X.shape)
print(Y.shape)

(165, 128, 128, 3)
(165,)


In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [ ]:
from sklearn.utils import shuffle
X, Y = shuffle(X, Y, random_state=42)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(256, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 822ms/step - accuracy: 0.6061 - loss: 1.2844 - val_accuracy: 0.8485 - val_loss: 0.4863
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.7955 - loss: 0.4781 - val_accuracy: 0.4848 - val_loss: 0.6093
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 844ms/step - accuracy: 0.7955 - loss: 0.4155 - val_accuracy: 0.8182 - val_loss: 0.5130
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 745ms/step - accuracy: 0.9318 - loss: 0.2260 - val_accuracy: 0.8788 - val_loss: 0.3704
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.9394 - loss: 0.2113 - val_accuracy: 0.8788 - val_loss: 0.3429
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 8s 760ms/step - accuracy: 0.9621 - loss: 0.1313 - val_accuracy: 0.8788 - val_loss: 0.3357
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 818ms/step - accuracy: 0.9621 - loss: 0.1024 - val_accuracy: 0.8788 - val_loss: 0.4905
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 745ms/step - accuracy: 0.9924 - loss: 0.0734 - val_accuracy: 0.8485 - val_loss: 0.453

In [ ]:
model.evaluate(x_test, y_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9091 - loss: 0.4478 


[0.44782543182373047, 0.9090909361839294]

In [ ]:
model.save("car_classifier.h5")
print("Model saved")

Model saved
